# Intermediate Models - Data Verification

Verify the intermediate dbt model transformations: era, tenure, clutch, and enriched datasets.

In [ ]:
import duckdb
from pathlib import Path

db_path = Path.cwd().parent / 'lebron_analytics.duckdb'
conn = duckdb.connect(str(db_path), read_only=True)
print(f'Connected: {db_path}')

## 1. Shots by Era

In [ ]:
result = conn.execute("""
	SELECT
		era,
		COUNT(*) as shots,
		SUM(is_shot_made::int) as made,
		ROUND(AVG(is_shot_made::int) * 100, 1) as fg_pct
	FROM main.int_shots_with_era
	GROUP BY era
	ORDER BY era
""").fetchall()

for era, shots, made, fg_pct in result:
	print(f'{era:20} {shots:>6,} shots | {made:>6,} made | {fg_pct:>5}%')

## 2. Shots by Tenure

In [ ]:
result = conn.execute("""
	SELECT
		tenure,
		MIN(season) as first_season,
		MAX(season) as last_season,
		COUNT(*) as shots,
		ROUND(AVG(is_shot_made::int) * 100, 1) as fg_pct
	FROM main.int_shots_with_tenure
	GROUP BY tenure, tenure_order
	ORDER BY tenure_order
""").fetchall()

for tenure, first, last, shots, fg_pct in result:
	print(f'{tenure:10} ({first} to {last}) {shots:>6,} shots | {fg_pct:>5}% FG')

## 3. Clutch Performance

In [ ]:
result = conn.execute("""
	SELECT
		CASE WHEN is_clutch_time THEN 'Clutch' ELSE 'Non-Clutch' END as situation,
		COUNT(*) as shots,
		SUM(points_scored) as total_points,
		ROUND(AVG(is_shot_made::int) * 100, 1) as fg_pct
	FROM main.int_shots_with_clutch
	GROUP BY is_clutch_time
	ORDER BY is_clutch_time DESC
""").fetchall()

for situation, shots, points, fg_pct in result:
	print(f'{situation:12} {shots:>6,} shots | {points:>6,} points | {fg_pct:>5}% FG')

## 4. Shot Distance by Era

In [ ]:
result = conn.execute("""
	SELECT
		era,
		shot_distance_category,
		COUNT(*) as shots,
		ROUND(AVG(is_shot_made::int) * 100, 1) as fg_pct
	FROM main.int_shots_enriched
	GROUP BY era, shot_distance_category
	ORDER BY era, shot_distance_category
""").fetchall()

current_era = None
for era, category, shots, fg_pct in result:
	if era != current_era:
		print(f'\n{era}:')
		current_era = era
	print(f'  {category:20} {shots:>6,} shots | {fg_pct:>5}% FG')

## 5. Enriched Dataset Summary

In [ ]:
result = conn.execute("""
	SELECT
		COUNT(*) as total_shots,
		COUNT(DISTINCT game_id) as games,
		COUNT(DISTINCT season) as seasons,
		COUNT(DISTINCT tenure) as tenures,
		COUNT(DISTINCT era) as eras
	FROM main.int_shots_enriched
""").fetchone()

print(f'Total shots: {result[0]:,}')
print(f'Games:       {result[1]:,}')
print(f'Seasons:     {result[2]}')
print(f'Tenures:     {result[3]}')
print(f'Eras:        {result[4]}')

conn.close()